In [1]:
# ============================================================
# ERIP - Customer Master Bronze Ingestion
# Notebook: nb_ingest_customer_master
# Purpose:
# 1. Read Customer Master source file
# 2. Validate data contract
# 3. Run data quality checks
# 4. Write Bronze Delta table
# 5. Log ingestion metadata and DQ results
# ============================================================

# ====================================================
# SECTION 1
# Pipeline Initialization
# ====================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

source_system = "Customer Master"
source_file_path = "Files/01_Bronze/customer_master/customer_master.csv"
target_table = "bronze_customer_master"
pipeline_name = "nb_ingest_customer_master"

run_start_time = datetime.now()

print("ERIP Customer Master ingestion started")
print(f"Source system: {source_system}")
print(f"Source file: {source_file_path}")
print(f"Target table: {target_table}")

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 3, Finished, Available, Finished, False)

ERIP Customer Master ingestion started
Source system: Customer Master
Source file: Files/01_Bronze/customer_master/customer_master.csv
Target table: bronze_customer_master


In [2]:
# ====================================================
# SECTION 2
# Read source CSV
# ====================================================

customer_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_file_path)
)

display(customer_df.limit(10))

print(f"Rows read: {customer_df.count()}")
print(f"Columns read: {len(customer_df.columns)}")

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cdc028c8-1f26-47ed-8684-a0dece84f091)

Rows read: 1000
Columns read: 26


In [3]:
# ============================================================
# SECTION 3 - DATA CONTRACT VALIDATION
# ============================================================

required_columns = [
    "customer_id",
    "customer_name",
    "legal_entity_id",
    "lei_code",
    "legal_entity_type",
    "customer_group_id",
    "parent_company_name",
    "industry_code",
    "industry_name",
    "nace_code",
    "country_code",
    "country",
    "region",
    "risk_country",
    "segment",
    "annual_revenue",
    "total_assets",
    "kyc_risk_rating",
    "esg_score",
    "esg_risk_band",
    "relationship_start_date",
    "onboarding_date",
    "relationship_manager",
    "customer_status",
    "source_system",
    "extract_date"
]

missing_columns = list(
    set(required_columns) -
    set(customer_df.columns)
)

if len(missing_columns) == 0:
    print("✓ Data Contract Validation Passed")
else:
    print("✗ Missing Columns:")
    print(missing_columns)

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 5, Finished, Available, Finished, False)

✓ Data Contract Validation Passed


In [4]:
# ============================================================
# SECTION 4 - THREE-LAYER DATA CONTRACT VALIDATION
# Layer 1: Schema Rules
# Layer 2: Business Rules
# Layer 3: Regulatory / Banking Rules
# ============================================================

validation_results = []

def add_validation_result(layer, rule_id, rule_name, failed_count):
    status = "PASS" if failed_count == 0 else "FAIL"
    validation_results.append({
        "pipeline_name": pipeline_name,
        "source_system": source_system,
        "target_table": target_table,
        "validation_layer": layer,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "failed_count": int(failed_count),
        "status": status,
        "validation_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

# ----------------------------
# Layer 1: Schema Rules
# ----------------------------
add_validation_result(
    "Schema",
    "SCHEMA_001",
    "All required columns must be present",
    len(missing_columns)
)

# ----------------------------
# Layer 2: Business Rules
# ----------------------------
total_rows = customer_df.count()

duplicate_customer_ids = total_rows - customer_df.select("customer_id").distinct().count()
null_customer_ids = customer_df.filter(col("customer_id").isNull()).count()
negative_annual_revenue = customer_df.filter(col("annual_revenue") < 0).count()
invalid_customer_status = customer_df.filter(
    ~col("customer_status").isin("Active", "Watchlist", "Closed")
).count()

add_validation_result("Business", "BUS_001", "Customer ID must be unique", duplicate_customer_ids)
add_validation_result("Business", "BUS_002", "Customer ID must not be null", null_customer_ids)
add_validation_result("Business", "BUS_003", "Annual revenue must not be negative", negative_annual_revenue)
add_validation_result("Business", "BUS_004", "Customer status must be valid", invalid_customer_status)

# ----------------------------
# Layer 3: Regulatory / Banking Rules
# ----------------------------
missing_lei = customer_df.filter(col("lei_code").isNull()).count()
missing_nace = customer_df.filter(col("nace_code").isNull()).count()
invalid_kyc = customer_df.filter(~col("kyc_risk_rating").isin("Low", "Medium", "High")).count()
invalid_esg_score = customer_df.filter((col("esg_score") < 0) | (col("esg_score") > 100)).count()

add_validation_result("Regulatory", "REG_001", "LEI code must be present", missing_lei)
add_validation_result("Regulatory", "REG_002", "NACE code must be present", missing_nace)
add_validation_result("Regulatory", "REG_003", "KYC risk rating must be valid", invalid_kyc)
add_validation_result("Regulatory", "REG_004", "ESG score must be between 0 and 100", invalid_esg_score)

validation_df = spark.createDataFrame(validation_results)

display(validation_df)

failed_validations = validation_df.filter(col("status") == "FAIL").count()

if failed_validations > 0:
    raise Exception(f"Data Contract Failed: {failed_validations} validation rule(s) failed.")
else:
    print("✓ Three-layer Data Contract Validation Passed")

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c01f2ebb-a02f-4777-b49d-9c0463c2fd6e)

✓ Three-layer Data Contract Validation Passed


In [5]:
# ============================================================
# SECTION 4 - VALIDATION FRAMEWORK
# Data Contract + Data Quality + Business + Regulatory Checks
# ============================================================

# Add ingestion audit columns to source data
customer_bronze_df = (
    customer_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit(pipeline_name))
    .withColumn("bronze_load_date", current_date())
)

# Write Bronze Delta table
customer_bronze_df.write.mode("overwrite").format("delta").saveAsTable("bronze_customer_master")

# Write validation results table
validation_df.write.mode("append").format("delta").saveAsTable("dq_validation_results")

print("✓ Bronze Delta table created: bronze_customer_master")
print("✓ DQ validation results written: dq_validation_results")
print(f"Rows written to Bronze: {customer_bronze_df.count()}")

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 7, Finished, Available, Finished, False)

✓ Bronze Delta table created: bronze_customer_master
✓ DQ validation results written: dq_validation_results
Rows written to Bronze: 1000


In [8]:
# ============================================================
# SECTION 5 - DATA QUALITY SUMMARY
# ============================================================

total_validation_rules = validation_df.count()
passed_validation_rules = validation_df.filter(col("status") == "PASS").count()
failed_validation_rules = validation_df.filter(col("status") == "FAIL").count()

dq_score = (passed_validation_rules / total_validation_rules) * 100

print("Data Quality Summary")
print("--------------------")
print(f"Total validation rules: {total_validation_rules}")
print(f"Passed validation rules: {passed_validation_rules}")
print(f"Failed validation rules: {failed_validation_rules}")
print(f"Data Quality Score: {dq_score}%")

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 10, Finished, Available, Finished, False)

Data Quality Summary
--------------------
Total validation rules: 9
Passed validation rules: 9
Failed validation rules: 0
Data Quality Score: 100.0%


In [10]:
# ============================================================
# SECTION 6 - METADATA LOGGING
# ============================================================

from datetime import datetime

run_end_time = datetime.now()
execution_time_seconds = (run_end_time - run_start_time).total_seconds()

metadata = [{
    "pipeline_name": pipeline_name,
    "source_system": source_system,
    "target_table": target_table,
    "rows_processed": customer_bronze_df.count(),
    "validation_rules": total_validation_rules,
    "dq_score": dq_score,
    "status": "SUCCESS",
    "run_start_time": run_start_time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_end_time": run_end_time.strftime("%Y-%m-%d %H:%M:%S"),
    "execution_time_seconds": execution_time_seconds
}]

metadata_df = spark.createDataFrame(metadata)

metadata_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("metadata_ingestion_log")

display(metadata_df)

print("✓ Metadata successfully written")

StatementMeta(, 235c3bf4-f462-4048-9a30-f33aa03dcef8, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26326780-7002-4064-a6aa-d1e4bc8acf34)

✓ Metadata successfully written
